# tokenmeter on a T4: three tenants through the metering proxy, then the cost dashboard

Qwen2.5-1.5B-Instruct on vLLM behind `tokenmeter proxy`. Three tenants with different shapes run concurrently for 8 minutes: `chat` (short prompts, 128-token answers, 3 req/s), `rag` (1,500-token prompts, 48-token answers, 0.5 req/s), `bulk` (64-token prompts, 384-token outputs, 0.5 req/s). Then `tokenmeter report` with the T4's price and the model's size. ~10 min install, ~12 min run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv

In [ ]:
# vLLM on the Kaggle image: the cu129 release wheel with the matching PyTorch index (the default wheel targets
# CUDA 13), and the image's preinstalled torchaudio/torchcodec removed because they are built for other CUDA
# versions and vLLM imports them (torchcodec is re-pulled by vLLM's requirements, hence the second uninstall).
%pip uninstall -q -y torchaudio torchcodec
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip uninstall -q -y torchcodec
%pip install -q "git+https://github.com/NomanJafar/servellm" "tokenbench[plots] @ git+https://github.com/NomanJafar/tokenbench"
# This repo is private: its wheel arrives as the attached Kaggle dataset tokenmeter-src (see portfolio automation).
!pip install -q /kaggle/input/tokenmeter-src/*.whl
import os; os.makedirs("/kaggle/working/results", exist_ok=True); os.chdir("/kaggle/working")
import torch, vllm, tokenmeter.costs, servellm.launch, tokenbench
assert torch.cuda.is_available()
import vllm.multimodal.media.audio
print("vllm", vllm.__version__, "torch", torch.__version__)

## Server, proxy, three tenants

In [ ]:
import json, subprocess, time, httpx
from servellm.launch import ServerConfig, VllmServer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"; UP = "http://127.0.0.1:8000"; PX = "http://127.0.0.1:8070"
server = VllmServer(ServerConfig(model=MODEL, dtype="float16", max_model_len=4096, gpu_memory_utilization=0.9), log_path="results/startup.log")
info = server.start(ready_timeout_s=1200); print(info)
proxy = subprocess.Popen(["tokenmeter", "proxy", "--upstream", UP, "--db", "results/tokenmeter.db", "--port", "8070"], stdout=open("results/proxy.log", "w"), stderr=subprocess.STDOUT)
for _ in range(100):
    try:
        if httpx.get(f"{PX}/health", timeout=1).status_code == 200: break
    except httpx.HTTPError: time.sleep(0.3)
# warm the engine before the metered window
subprocess.run(["tokenbench", "sweep", "--base-url", UP, "--model", MODEL, "--out", "results/_warmup.json", "--levels", "1,8", "--num-requests", "16",
                "--warmup", "0", "--prompt-tokens", "64", "--max-tokens", "64", "--request-timeout", "60"], check=True, capture_output=True, timeout=300)

In [ ]:
DUR = "480"
common = ["tokenbench", "sweep", "--base-url", PX, "--model", MODEL, "--mode", "open", "--duration", DUR, "--warmup", "0", "--prompts", "unique", "--request-timeout", "120"]
tenants = {
    "chat": ["--levels", "3",   "--prompt-tokens", "96",   "--max-tokens", "128", "--header", "x-tenant:chat"],
    "rag":  ["--levels", "0.5", "--prompt-tokens", "1500", "--max-tokens", "48",  "--header", "x-tenant:rag"],
    "bulk": ["--levels", "0.5", "--prompt-tokens", "64",   "--max-tokens", "384", "--header", "x-tenant:bulk"],
}
t_start = time.time()
procs = {t: subprocess.Popen(common + a + ["--out", f"results/load_{t}.json"], stdout=open(f"results/load_{t}.log", "w"), stderr=subprocess.STDOUT) for t, a in tenants.items()}
for t, p in procs.items():
    try: p.wait(timeout=int(DUR) + 300)
    except subprocess.TimeoutExpired: p.kill()
wall_h = (time.time() - t_start) / 3600
print(f"load done in {wall_h*60:.1f} min")
proxy.terminate(); server.stop()
json.dump({"wall_hours": wall_h, "startup": {"model_load_gib": info.model_load_gib, "kv_cache_tokens": info.kv_cache_tokens}}, open("results/run_meta.json", "w"))

## Report

T4 price: $0.35/h (typical on-demand cloud price for a T4 in 2026); hosted comparison at $0.15 / $0.60 per M in/out (a small-model API tier). The decode step time for bandwidth utilisation comes from the chat tenant's measured ITL.

In [ ]:
chat = json.load(open("results/load_chat.json"))["levels"][0]
step_ms = chat["itl"]["p50"] * 1000
!tokenmeter report --db results/tokenmeter.db --out results/dashboard --gpu-usd-per-hour 0.35 --params 1.54e9 --weight-gib {info.model_load_gib} --peak-tflops 65 --peak-gb-s 320 --prompt-weight 0.25 --hosted-in 0.15 --hosted-out 0.60 --gpu-hours {wall_h} --step-ms {step_ms}
from IPython.display import HTML, display
display(HTML(open("results/dashboard/dashboard.html").read()))

In [ ]:
!cd results && zip -qr ../results.zip . && ls -la ../results.zip